# Probability Measures -- Concept 19

A **probability measure** is a function $P : \mathcal{F} \to [0,1]$ on a $\sigma$-algebra $\mathcal{F}$ over a sample space $\Omega$ satisfying the **Kolmogorov axioms**:

- **(K1)** $P(A) \geq 0$ for every $A \in \mathcal{F}$,
- **(K2)** $P(\Omega) = 1$,
- **(K3)** $P(\bigsqcup_n A_n) = \sum_n P(A_n)$ for pairwise disjoint $\{A_n\}$.

This notebook builds a discrete probability measure on $\Omega = \{1, \dots, 6\}$ (a loaded die), verifies (K1)--(K3) on the full power-set $\sigma$-algebra, and confirms the inclusion--exclusion identity $P(A \cup B) = P(A) + P(B) - P(A \cap B)$.

## 1. Setup: sample space, weights, sigma-algebra

We take $\Omega = \{1, \dots, 6\}$ with weights $w_1 = \dots = w_5 = 1/7$ and $w_6 = 2/7$ (a loaded die). The $\sigma$-algebra is the full power set $2^{\Omega}$, which has $2^6 = 64$ elements. We use `Fraction` to keep all arithmetic exact.

In [ ]:
from fractions import Fraction
from itertools import combinations

omega = frozenset({1, 2, 3, 4, 5, 6})
weights = {i: Fraction(1, 7) for i in range(1, 6)}
weights[6] = Fraction(2, 7)

def power_set(s):
    items = list(s)
    return [frozenset(c) for r in range(len(items) + 1) for c in combinations(items, r)]

sigma_algebra = power_set(omega)
print(f'|Omega| = {len(omega)}, |sigma-algebra| = {len(sigma_algebra)}')
print('weights:', {k: str(v) for k, v in sorted(weights.items())})

## 2. Define the probability measure

Given weights $w_i \geq 0$ with $\sum_i w_i = 1$, we define $P : 2^{\Omega} \to [0,1]$ by
$$P(A) = \sum_{i \in A} w_i.$$
(K1) and (K2) are immediate from non-negativity and normalisation of the weights; (K3) on a finite space reduces to finite additivity, which holds because sums over disjoint index sets split.

In [ ]:
def P(A):
    return sum(weights[x] for x in A)

# Spot-check a few events
print('P(empty)        =', P(frozenset()))
print('P(Omega)        =', P(omega))
print('P({1,3,5}) odd  =', P(frozenset({1, 3, 5})))
print('P({6})          =', P(frozenset({6})))

## 3. Verify the three Kolmogorov axioms

We check (K1) on every event, (K2) on $\Omega$, and (K3) by testing finite additivity on every disjoint pair $(A, B)$ in the $\sigma$-algebra. On a finite space this is the strongest statement available (and it implies the countable version, since all but finitely many $A_n$ must be empty).

In [ ]:
k1 = all(P(A) >= 0 for A in sigma_algebra)
k2 = (P(omega) == 1)

k3 = True
for A, B in combinations(sigma_algebra, 2):
    if A & B:
        continue
    if P(A | B) != P(A) + P(B):
        k3 = False
        break

print(f'(K1) non-negativity    : {k1}')
print(f'(K2) P(Omega) = 1      : {k2}')
print(f'(K3) finite additivity : {k3}')
assert k1 and k2 and k3

## 4. Inclusion-exclusion on two events

Let $A = \{1, 3, 5\}$ ("roll is odd") and $B = \{4, 5, 6\}$ ("roll is at least 4"). We verify
$$P(A \cup B) = P(A) + P(B) - P(A \cap B).$$
Note that $A$ and $B$ overlap at $\{5\}$, so naive addition would double-count.

In [ ]:
A = frozenset({1, 3, 5})
B = frozenset({4, 5, 6})

lhs = P(A | B)
rhs = P(A) + P(B) - P(A & B)

print(f'P(A)            = {P(A)}')
print(f'P(B)            = {P(B)}')
print(f'P(A inter B)    = {P(A & B)}')
print(f'P(A union B)    = {lhs}')
print(f'sum - intersect = {rhs}')
assert lhs == rhs
print('inclusion-exclusion verified.')

## 5. Continuity from below (sanity check on a nested sequence)

Continuity from below says $A_n \uparrow A \Rightarrow P(A_n) \to P(A)$. On a finite space this is trivial -- the sequence stabilises in finitely many steps -- but it is still a useful sanity check. Take $A_n = \{1, 2, \dots, \min(n, 6)\}$; then $A_n \uparrow \Omega$ and $P(A_n) \to 1$.

In [ ]:
for n in range(1, 9):
    A_n = frozenset(range(1, min(n, 6) + 1))
    print(f'n = {n}: A_n = {sorted(A_n)}, P(A_n) = {P(A_n)}')
print()
print('P(A_n) increases to P(Omega) = 1, as continuity from below predicts.')

## 6. Takeaways

- A probability measure is determined on a finite/discrete $\Omega$ by its weights on singletons; on uncountable spaces (e.g. $[0,1]$ with Lebesgue measure) singletons can carry zero mass and the structure is richer.
- The Kolmogorov axioms (K1)--(K3) are exactly the structure needed to derive every standard probability identity: complement, monotonicity, inclusion--exclusion, Boole's inequality, continuity from below/above.
- In ML papers, advantage normalisation, importance ratios, and concentration bounds all implicitly invoke (K1)--(K3); see GRPO (Akgül et al. 2026, §3.1) where the centred advantage has expectation zero precisely because the underlying Bernoulli measure is normalised.